In [1]:
import pandas as pd
import kagglehub
import ast
import numpy as np
from collections import Counter
from unidecode import unidecode
import pycountry

In [2]:
path_ml = kagglehub.dataset_download("grouplens/movielens-latest-full")
path_tmdb = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("MovieLens path:", path_ml)
print("TMDB path:", path_tmdb)

# MovieLens files
ratings_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv')
movies_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv')
links_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv')
tags_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv')

# TMDB files
metadata_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv', low_memory=False)
credits_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv')
keywords_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv')

MovieLens path: /Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1
TMDB path: /Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7


In [3]:
def get_director(crew):
    return next((p['name'] for p in crew if p.get('job') == 'Director'), None)

def get_producer(crew):
    return next((p['name'] for p in crew if p.get('job') == 'Producer'), None)

def get_lead_actor(cast):
    return next((p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf'))) if 'name' in p), None)

def get_gender_of_lead(cast, lead_name):
    return next((p['gender'] for p in cast if p.get('name') == lead_name), None)

def get_other_lead(cast, lead_name):
    lead_gender = get_gender_of_lead(cast, lead_name)
    opposite_gender = {1: 2, 2: 1}.get(lead_gender)
    return next(
        (p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf')))
         if p.get('name') != lead_name and p.get('gender') == opposite_gender),
        None
    )

def get_other_actors(cast, exclude_names, max_count=3):
    return [
        p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf')))
        if p.get('name') not in exclude_names and p.get('name') is not None
    ][:max_count]


In [4]:
def iso_639_to_name(code):
    try:
        return pycountry.languages.get(alpha_2=code).name
    except:
        return code.upper()

def safe_language_name(d):
    if isinstance(d, dict):
        name = d.get('name')
        code = d.get('iso_639_1')
        if name and '?' not in name:
            return unidecode(name)
        if code:
            return iso_639_to_name(code)
    return None


In [5]:
# Clean links
links_ml = links_ml[links_ml['tmdbId'].notnull()]
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)

# Clean metadata
metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()]
metadata_tmdb['id'] = metadata_tmdb['id'].astype(int)
metadata_tmdb['genres'] = metadata_tmdb['genres'].fillna('[]').apply(ast.literal_eval)

# Parse credits
credits_tmdb['cast'] = credits_tmdb['cast'].apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].apply(ast.literal_eval)
credits_tmdb['tmdbId'] = credits_tmdb['id']
credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['producer'] = credits_tmdb['crew'].apply(get_producer)
credits_tmdb['lead_actor'] = credits_tmdb['cast'].apply(get_lead_actor)
credits_tmdb['other_lead'] = credits_tmdb.apply(
    lambda row: get_other_lead(row['cast'], row['lead_actor']), axis=1
)
credits_tmdb['other_actors'] = credits_tmdb.apply(
    lambda row: get_other_actors(row['cast'], exclude_names={row['lead_actor'], row['other_lead']}), axis=1
)

# Parse keywords
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)])
keywords_tmdb['tmdbId'] = keywords_tmdb['id']

# Aggregate tags
tags_agg = tags_ml.groupby('movieId')['tag'].apply(lambda x: list(set(x))).reset_index()

# Ratings stats
rating_stats = ratings_ml.groupby('movieId')['rating'].agg(['mean', 'min', 'max', 'count']).reset_index()
rating_stats.columns = ['movieId', 'vote_average', 'vote_min', 'vote_max', 'vote_count']

In [6]:
movies_ml_links = pd.merge(movies_ml, links_ml, on='movieId')
metadata_tmdb = metadata_tmdb.rename(columns={'id': 'tmdbId'})
movies_full = pd.merge(movies_ml_links, metadata_tmdb, on='tmdbId', how='inner')

movies_full = pd.merge(movies_full, credits_tmdb[['tmdbId', 'director', 'producer', 'other_lead', 'other_actors','lead_actor']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, keywords_tmdb[['tmdbId', 'keywords']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, tags_agg, on='movieId', how='left')
movies_full = pd.merge(movies_full, rating_stats, on='movieId', how='left')

In [7]:
# Runtime binning
bin_edges = list(range(0, 301, 30)) + [np.inf]
labels = [f'{bin_edges[i]}–{bin_edges[i+1]}min' if bin_edges[i+1] != np.inf else f'{bin_edges[i]}min+' for i in range(len(bin_edges) - 1)]
movies_full['runtime_bin'] = pd.cut(movies_full['runtime'], bins=bin_edges, labels=labels)

# Release dates
movies_full['release_date_parsed'] = pd.to_datetime(movies_full['release_date'], errors='coerce')
release_year_tmdb = movies_full['release_date_parsed'].dt.year
release_year_ml = movies_full['title_x'].str.extract(r'\((\d{4})\)')[0].astype(float)

movies_full['release_year_tmdb'] = release_year_tmdb
movies_full['release_year_ml'] = release_year_ml
movies_full['release_year'] = release_year_tmdb.combine_first(release_year_ml)
movies_full['release_year_merged'] = movies_full[['release_year_tmdb', 'release_year_ml']].min(axis=1)

# Ratings
movies_full['vote_count'] = movies_full[['vote_count_x', 'vote_count_y']].max(axis=1)
movies_full['vote_average'] = movies_full.apply(
    lambda row: row['vote_average_x'] if row['vote_count_x'] >= row['vote_count_y'] else row['vote_average_y'],
    axis=1
)

# Title
movies_full['title'] = movies_full['title_y'].combine_first(movies_full['title_x'])

# Genres
movies_full['genres_x_list'] = movies_full['genres_x'].fillna('').apply(lambda x: x.split('|') if isinstance(x, str) else [])
movies_full['genres_y_list'] = movies_full['genres_y'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)] if isinstance(x, list) else [])
movies_full['genre_list'] = movies_full.apply(lambda row: sorted(set(row['genres_x_list']) | set(row['genres_y_list'])), axis=1)

# Main genre (prefer TMDB, fallback to MovieLens)
def extract_main_genre(genres_y): return genres_y[0]['name'] if isinstance(genres_y, list) and len(genres_y) > 0 and isinstance(genres_y[0], dict) else None
movies_full['main_genre'] = movies_full['genres_y'].apply(extract_main_genre)
movies_full['main_genre'] = movies_full.apply(lambda row: row['main_genre'] if pd.notnull(row['main_genre']) else (row['genres_x_list'][0] if row['genres_x_list'] else None), axis=1)

# Additional features
movies_full['release_month'] = movies_full['release_date_parsed'].dt.month
movies_full['release_decade'] = (movies_full['release_year'] // 10) * 10
movies_full['popularity_score'] = movies_full['vote_average'] * np.log1p(movies_full['vote_count'])

In [8]:
all_genres = movies_full['genre_list'].explode()
genre_counts = Counter(all_genres)
genre_count_df = pd.DataFrame(genre_counts.items(), columns=['genre', 'count']).sort_values(by='count', ascending=False)
genre_count_df.head(10)  # Top 10 genres

,genre,count
7,Drama,23095
3,Comedy,14632
10,Thriller,8965
6,Romance,8238
8,Action,7554
9,Crime,5399
11,Horror,5073
17,Documentary,4415
0,Adventure,4412
13,Mystery,3205


In [9]:
movies_full = movies_full.drop(columns=[
    # 'title_x', 'title_y',
    # 'vote_average_x', 'vote_average_y',
    # 'vote_count_x', 'vote_count_y',
    # 'release_year_tmdb', 'release_year_ml',
    'release_date',
    # 'genres_x', 'genres_y'
])

movies_full = movies_full.rename(columns={'release_date_parsed': 'release_date'})

In [10]:
pd.set_option('display.max_columns', None)

In [11]:
movies_full["production_countries"][3]

"[{'iso_3166_1': 'US', 'name': 'United States of America'}]"

In [12]:
movies_full["production_countries"].unique()

array(["[{'iso_3166_1': 'US', 'name': 'United States of America'}]",
       "[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]",
       "[{'iso_3166_1': 'GB', 'name': 'United Kingdom'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]",
       ...,
       "[{'iso_3166_1': 'PL', 'name': 'Poland'}, {'iso_3166_1': 'CZ', 'name': 'Czech Republic'}, {'iso_3166_1': 'SK', 'name': 'Slovakia'}]",
       "[{'iso_3166_1': 'CU', 'name': 'Cuba'}, {'iso_3166_1': 'DE', 'name': 'Germany'}, {'iso_3166_1': 'ES', 'name': 'Spain'}]",
       "[{'iso_3166_1': 'EG', 'name': 'Egypt'}, {'iso_3166_1': 'IT', 'name': 'Italy'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]"],
      shape=(2391,), dtype=object)

In [13]:
movies_full.columns

Index(['movieId', 'title_x', 'genres_x', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_y', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline',
       'title_y', 'video', 'vote_average_x', 'vote_count_x', 'director',
       'producer', 'other_lead', 'other_actors', 'lead_actor', 'keywords',
       'tag', 'vote_average_y', 'vote_min', 'vote_max', 'vote_count_y',
       'runtime_bin', 'release_date', 'release_year_tmdb', 'release_year_ml',
       'release_year', 'release_year_merged', 'vote_count', 'vote_average',
       'title', 'genres_x_list', 'genres_y_list', 'genre_list', 'main_genre',
       'release_month', 'release_decade', 'popularity_score'],
      dtype='object')

In [14]:
movies_full["spoken_languages"].unique()

array(["[{'iso_639_1': 'en', 'name': 'English'}]",
       "[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",
       "[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'es', 'name': 'Español'}]",
       ...,
       "[{'iso_639_1': 'sv', 'name': 'svenska'}, {'iso_639_1': 'de', 'name': 'Deutsch'}]",
       "[{'iso_639_1': 'ar', 'name': 'العربية'}, {'iso_639_1': 'pl', 'name': 'Polski'}]",
       "[{'iso_639_1': 'ff', 'name': 'Fulfulde'}, {'iso_639_1': 'en', 'name': 'English'}]"],
      shape=(1932,), dtype=object)

In [15]:
movies_full['production_countries_parsed'] = movies_full['production_countries'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else []
)

movies_full['spoken_languages_parsed'] = movies_full['spoken_languages'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else []
)

# movies_full['production_country_names'] = movies_full['production_countries_parsed'].apply(
#     lambda lst: [d['name'] for d in lst if isinstance(d, dict) and 'name' in d]
# )

# movies_full['spoken_language_names'] = movies_full['spoken_languages_parsed'].apply(
#     lambda lst: [d['name'] for d in lst if isinstance(d, dict) and 'name' in d]
# )

movies_full['production_country_names'] = movies_full['production_countries_parsed'].apply(
    lambda lst: [unidecode(d['name']) for d in lst if isinstance(d, dict) and 'name' in d]
)

# Fix: spoken_language_names
movies_full['spoken_language_names'] = movies_full['spoken_languages_parsed'].apply(
    lambda lst: [safe_language_name(d) for d in lst if isinstance(d, dict)]
)

In [16]:
# movies_full['main_language'] = movies_full['spoken_language_names'].apply(
#     lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
# )
# movies_full['main_country'] = movies_full['production_country_names'].apply(
#     lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
# )

movies_full['main_language'] = movies_full['spoken_language_names'].apply(
    lambda x: unidecode(x[0]) if isinstance(x, list) and len(x) > 0 else None
)

# Main production country with transliteration
movies_full['main_country'] = movies_full['production_country_names'].apply(
    lambda x: unidecode(x[0]) if isinstance(x, list) and len(x) > 0 else None
)

In [17]:
# Show a few raw entries
movies_full[movies_full['main_country'].str.contains(r'\?{2,}', na=False)][['title', 'spoken_language_names']]


,title,spoken_language_names


In [18]:
movies_full['has_translation'] = movies_full.apply(
    lambda row: [lang for lang in row['spoken_language_names'] if lang != row['main_language']]
    if isinstance(row['spoken_language_names'], list) else [],
    axis=1
)

movies_full['has_translation']

0                   []
1           [Francais]
2                   []
3                   []
4                   []
             ...      
46893               []
46894    [hindii, rdw]
46895               []
46896               []
46897               []
Name: has_translation, Length: 46898, dtype: object

In [19]:
movies_full.iloc[0]["belongs_to_collection"]

"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}"

In [20]:
movies_full['collection_name'] = movies_full['belongs_to_collection'].apply(
    lambda x: ast.literal_eval(x).get('name') if pd.notnull(x) else None
)
movies_full.loc[movies_full['collection_name'].notnull(), ['title', 'collection_name']].head()

,title,collection_name
0,Toy Story,Toy Story Collection
2,Grumpier Old Men,Grumpy Old Men Collection
4,Father of the Bride Part II,Father of the Bride Collection
9,GoldenEye,James Bond Collection
12,Balto,Balto Collection


In [21]:
movies_full.columns

Index(['movieId', 'title_x', 'genres_x', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_y', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline',
       'title_y', 'video', 'vote_average_x', 'vote_count_x', 'director',
       'producer', 'other_lead', 'other_actors', 'lead_actor', 'keywords',
       'tag', 'vote_average_y', 'vote_min', 'vote_max', 'vote_count_y',
       'runtime_bin', 'release_date', 'release_year_tmdb', 'release_year_ml',
       'release_year', 'release_year_merged', 'vote_count', 'vote_average',
       'title', 'genres_x_list', 'genres_y_list', 'genre_list', 'main_genre',
       'release_month', 'release_decade', 'popularity_score',
       'production_countries_parsed', 'spoken_languages_parsed',
       'production_country_names', 'spoken_language_names', 'm

In [22]:

movies_full.iloc[0]["production_companies"]


"[{'name': 'Pixar Animation Studios', 'id': 3}]"

In [23]:
movies_full['production_companies_parsed'] = movies_full['production_companies'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else []
)

# Step 2: Extract list of company names
movies_full['production_company_names'] = movies_full['production_companies_parsed'].apply(
    lambda lst: [d['name'] for d in lst if isinstance(d, dict) and 'name' in d]
)

# Step 3: Get main/first company
movies_full['main_production_company'] = movies_full['production_company_names'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

In [24]:
movies_full['production_company_names']

0                                [Pixar Animation Studios]
1        [TriStar Pictures, Teitler Film, Interscope Co...
2                           [Warner Bros., Lancaster Gate]
3                 [Twentieth Century Fox Film Corporation]
4             [Sandollar Productions, Touchstone Pictures]
                               ...                        
46893                                                   []
46894                             [Aamir Khan Productions]
46895    [Creative Entertainment Group, Silver Bullet P...
46896                                                   []
46897                                 [sabotage film GmbH]
Name: production_company_names, Length: 46898, dtype: object

In [25]:
movies_full.columns

Index(['movieId', 'title_x', 'genres_x', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_y', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline',
       'title_y', 'video', 'vote_average_x', 'vote_count_x', 'director',
       'producer', 'other_lead', 'other_actors', 'lead_actor', 'keywords',
       'tag', 'vote_average_y', 'vote_min', 'vote_max', 'vote_count_y',
       'runtime_bin', 'release_date', 'release_year_tmdb', 'release_year_ml',
       'release_year', 'release_year_merged', 'vote_count', 'vote_average',
       'title', 'genres_x_list', 'genres_y_list', 'genre_list', 'main_genre',
       'release_month', 'release_decade', 'popularity_score',
       'production_countries_parsed', 'spoken_languages_parsed',
       'production_country_names', 'spoken_language_names', 'm

In [26]:
movies_full

,movieId,title_x,genres_x,imdbId,tmdbId,adult,belongs_to_collection,budget,genres_y,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,revenue,runtime,spoken_languages,status,tagline,title_y,video,vote_average_x,vote_count_x,director,producer,other_lead,other_actors,lead_actor,keywords,tag,vote_average_y,vote_min,vote_max,vote_count_y,runtime_bin,release_date,release_year_tmdb,release_year_ml,release_year,release_year_merged,vote_count,vote_average,title,genres_x_list,genres_y_list,genre_list,main_genre,release_month,release_decade,popularity_score,production_countries_parsed,spoken_languages_parsed,production_country_names,spoken_language_names,main_language,main_country,has_translation,collection_name,production_companies_parsed,production_company_names,main_production_company
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,John Lasseter,Bonnie Arnold,Annie Potts,"[Tim Allen, Don Rickles, Jim Varney]",Tom Hanks,"[jealousy, toy, boy, friendship, friends, riva...","[ss, witty, Buzz Lightyear, martial arts, Disn...",3.886649,0.5,5.0,68469.0,60–90min,1995-10-30,1995.0,1995.0,1995.0,1995.0,68469.0,3.886649,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]","[Animation, Comedy, Family]","[Adventure, Animation, Children, Comedy, Famil...",Animation,10.0,1990.0,43.274542,"[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]",[United States of America],[English],English,United States of America,[],Toy Story Collection,"[{'name': 'Pixar Animation Studios', 'id': 3}]",[Pixar Animation Studios],Pixar Animation Studios
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,Joe Johnston,Scott Kroopf,Kirsten Dunst,"[Jonathan Hyde, Bradley Pierce, Bonnie Hunt]",Robin Williams,"[board game, disappearance, based on children'...","[board game, Saturn Award (Best Supporting Act...",3.246583,0.5,5.0,27143.0,90–120min,1995-12-15,1995.0,1995.0,1995.0,1995.0,27143.0,3.246583,Jumanji,"[Adventure, Children, Fantasy]","[Adventure, Fantasy, Family]","[Adventure, Children, Family, Fantasy]",Adventure,12.0,1990.0,33.144077,"[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",[United States of America],"[English, Francais]",English,United States of America,[Francais],None,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[TriStar Pictures, Teitler Film, Interscope Co...",TriStar Pictures
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,Howard Deutch,None,Ann-Margret,"[Jac

In [27]:
movies_full.columns

Index(['movieId', 'title_x', 'genres_x', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_y', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline',
       'title_y', 'video', 'vote_average_x', 'vote_count_x', 'director',
       'producer', 'other_lead', 'other_actors', 'lead_actor', 'keywords',
       'tag', 'vote_average_y', 'vote_min', 'vote_max', 'vote_count_y',
       'runtime_bin', 'release_date', 'release_year_tmdb', 'release_year_ml',
       'release_year', 'release_year_merged', 'vote_count', 'vote_average',
       'title', 'genres_x_list', 'genres_y_list', 'genre_list', 'main_genre',
       'release_month', 'release_decade', 'popularity_score',
       'production_countries_parsed', 'spoken_languages_parsed',
       'production_country_names', 'spoken_language_names', 'm

In [28]:
movies_full = movies_full[[
    # 1. Movie & genre
    'title', 'main_genre', 'genre_list', 'keywords', 'tag', 'collection_name',

    # 2. Cast & crew
    'director', 'producer', 'lead_actor', 'other_lead',
    'other_actors',

    # 3. Production
    'main_production_company', 'production_company_names',
    'main_country', 'production_country_names',

    # 4. Language
    'main_language', 'spoken_language_names', 'has_translation',

    # 5. Temporal
    'release_date', 'release_year', 'release_month', 'release_decade',
    'runtime', 'runtime_bin',

    # 6. Ratings
    'vote_average', 'vote_count', 'vote_min', 'vote_max', 'popularity_score',

    # 7. Financials
    'budget', 'revenue',

    # 8. IDs
    'movieId', 'imdbId', 'tmdbId', 'imdb_id'
]]


In [31]:
movies_full.to_parquet("../processed_data/movies_data_updated.parquet", index = False)